In [1]:
import sys
sys.path.append(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq")

from TCC.utils.constantes import *

In [2]:
# df_btc_price = pd.read_csv(rf"/home/baia/z/git/data-science-analytics/TCC/data/dados_btc/raw/price_btc.csv")
df_btc_price = pd.read_csv(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq/TCC/data/dados_btc/raw/price_btc.csv")
df_btc_price['Data_UTC'] = pd.to_datetime(df_btc_price['time'], unit='s', utc=True,).dt.strftime("%Y-%m-%d")

df_sp500 = (pd.read_csv(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq/TCC/data/dados_macro/raw/2014_SP500_PRICE.csv")

                         .assign(Data_UTC = lambda df: pd.to_datetime(df['time'], utc=True))
                         .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))

                        .rename(columns={'close': 'close_price'}) 

                        [['Data_UTC', 'close_price']] 
 
)

df_sp500_log_Ret =(
    df_periodo
        .merge(df_sp500, how='left', on='Data_UTC')
        # Preço se mantém ao FDS
        .assign(close_price = lambda df: df['close_price'].fillna(method='ffill'))
        .assign(SPX_Log_Ret = lambda df: np.log(df['close_price']) - np.log(df['close_price'].shift(1)))


        .query("Data_UTC > '2017-01-03'")
        [['Data_UTC','close_price','SPX_Log_Ret']]

)
df_sp500_log_Ret

df_vix = (pd.read_csv(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq/TCC/data/dados_macro/raw/201501_VIX.csv")

                        .assign(Data_UTC = lambda df: pd.to_datetime(df['time'],unit='s', utc=True))
                        .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))
                        .rename(columns={'close': 'close_price'}) 

                        [['Data_UTC', 'close_price']] 
 
)
df_vix

df_vix_log_ret =(
    df_periodo
        .merge(df_vix, how='left', on='Data_UTC')
        .assign(close_price = lambda df: df['close_price'].ffill())
        .assign(VIX_Log_Ret = lambda df: np.log(df['close_price']) - np.log(df['close_price'].shift(1)))

        .query("Data_UTC > '2017-01-03'")
        [['Data_UTC','close_price','VIX_Log_Ret']]

)
df_vix_log_ret

df_us10y = (pd.read_csv(rf"/Users/baia/Desktop/PYTHON/mba_dsa_usp_esalq/TCC/data/dados_macro/raw/201501_US10Y.csv")

                        .assign(Data_UTC = lambda df: pd.to_datetime(df['time'],unit='s', utc=True))
                        .assign(Data_UTC = lambda df: df['Data_UTC'].dt.strftime("%Y-%m-%d"))
                        .rename(columns={'close': 'close_price'}) 

                        [['Data_UTC', 'close_price']] 
 
)
df_us10y

df_us10y_diff =(
    df_periodo
        .merge(df_us10y, how='left', on='Data_UTC')
        .assign(close_price = lambda df: df['close_price'].ffill())
        .assign(US10Y_Diff = lambda df: df['close_price'].diff())

        .query("Data_UTC > '2017-01-03'")
        [['Data_UTC','close_price','US10Y_Diff']]

)
df_us10y_diff

/var/folders/h1/_3z4z04x4j17zlfj224w53hr0000gn/T/ipykernel_4453/4063713309.py:20: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  .assign(close_price = lambda df: df['close_price'].fillna(method='ffill'))


,Data_UTC,close_price,US10Y_Diff
4,2017-01-04,2.4413,-0.0059
5,2017-01-05,2.3479,-0.0934
6,2017-01-06,2.4221,0.0742
7,2017-01-07,2.4221,0.0000
8,2017-01-08,2.4221,0.0000
...,...,...,...
3131,2025-07-28,4.3240,-0.0900
3132,2025-07-29,4.3720,0.0480
3133,2025-07-30,4.3820,0.0100
3134,2025-07-31,4.2160,-0.1660


In [6]:
# Base dataframe
df_final_feature = df_periodo.copy()

# Left join com df_btc_price
df_final_feature = df_final_feature.merge(
    df_btc_price[['Data_UTC', 'close']],
    on='Data_UTC',
    how='left'
).rename(columns={
    'close': 'btc_close'
})

# Left join com df_sp500_log_Ret
df_final_feature = df_final_feature.merge(
    df_sp500_log_Ret,
    on='Data_UTC',
    how='left'
).rename(columns={'close_price': 'sp500_close'})

# Left join com df_vix_log_ret
df_final_feature = df_final_feature.merge(
    df_vix_log_ret,
    on='Data_UTC',
    how='left'
).rename(columns={'close_price': 'vix_close'})

# Left join com df_us10y_diff
df_final_feature = df_final_feature.merge(
    df_us10y_diff,
    on='Data_UTC',
    how='left'
).rename(columns={'close_price': 'us10y_close'})

df_final_feature = df_final_feature.ffill().query("Data_UTC >= '2017-01-04'").reset_index(drop=True)

df_final_feature = df_final_feature.assign(Data_UTC=pd.to_datetime(df_final_feature['Data_UTC']))

df_final_feature = df_final_feature.assign( btc_Log_Ret = lambda df: np.log(df['btc_close']) - np.log(df['btc_close'].shift(1)))
df_varejo = df_final_feature.loc[:DATA_DIVISAO_ERA]

In [7]:
from statsmodels.tsa.stattools import grangercausalitytests
import warnings
warnings.filterwarnings('ignore')

# ==============================================================================
# 4. TESTE DE CAUSALIDADE DE GRANGER (Para a Seção 3.4)
# ==============================================================================

def rodar_granger_resumido(df, target, features, max_lag=3):
    resultados = []
    
    for feature in features:
        # Prepara os dados (Target na coluna 1, Causa Potencial na coluna 2)
        # H0: A Coluna 2 NÃO causa a Coluna 1
        data_test = df[[target, feature]].dropna()
        
        try:
            # Roda o teste
            test_result = grangercausalitytests(data_test, maxlag=[max_lag], verbose=False)
            
            # Extrai o P-Valor (do teste SSR base F-test)
            p_value = test_result[max_lag][0]['ssr_ftest'][1]
            
            resultados.append({
                'Causa (X)': feature,
                'Efeito (Y)': target,
                'Lag (Dias)': max_lag,
                'P-Valor': round(p_value, 5),
                'Resultado': '✅ Causalidade Significante' if p_value < 0.05 else '❌ Sem Causalidade'
            })
        except:
            pass
            
    return pd.DataFrame(resultados)

# Configuração
TARGET_FINAL = 'btc_Log_Ret'
FEATURES_MACRO = ['SPX_Log_Ret', 'VIX_Log_Ret', 'US10Y_Diff']

# 1. Teste na Era Varejo
print(f"\n>>> RESULTADO GRANGER: ERA VAREJO (Pré-{DATA_DIVISAO_ERA}) <<<")
df_res_varejo = rodar_granger_resumido(df_varejo, TARGET_FINAL, FEATURES_MACRO, max_lag=2)
print(df_res_varejo)

# 2. Teste na Era Institucional
print(f"\n>>> RESULTADO GRANGER: ERA INSTITUCIONAL (Pós-{DATA_DIVISAO_ERA}) <<<")
# Precisamos pegar as colunas do df original filtrado por data
df_inst_features = df_final_feature.loc[DATA_DIVISAO_ERA:][FEATURES_MACRO + [TARGET_FINAL]]
df_res_inst = rodar_granger_resumido(df_inst_features, TARGET_FINAL, FEATURES_MACRO, max_lag=2)
print(df_res_inst)


>>> RESULTADO GRANGER: ERA VAREJO (Pré-2020-03-11) <<<
     Causa (X)   Efeito (Y)  Lag (Dias)  P-Valor          Resultado
0  SPX_Log_Ret  btc_Log_Ret           2  0.45183  ❌ Sem Causalidade
1  VIX_Log_Ret  btc_Log_Ret           2  0.76491  ❌ Sem Causalidade
2   US10Y_Diff  btc_Log_Ret           2  0.64563  ❌ Sem Causalidade

>>> RESULTADO GRANGER: ERA INSTITUCIONAL (Pós-2020-03-11) <<<
     Causa (X)   Efeito (Y)  Lag (Dias)  P-Valor          Resultado
0  SPX_Log_Ret  btc_Log_Ret           2  0.20291  ❌ Sem Causalidade
1  VIX_Log_Ret  btc_Log_Ret           2  0.74600  ❌ Sem Causalidade
2   US10Y_Diff  btc_Log_Ret           2  0.13701  ❌ Sem Causalidade
